In [0]:
SELECT t.transaction_id, t.store_id, t.basket_value_zar, s.province, s.store_format
FROM freshmart_analysis.default.fresh_mart_transactiondataset_1 t
JOIN freshmart_analysis.default.fresh_mart_storesdataset_1 s ON t.store_id = s.store_id
LIMIT 5;

--checking Decline by store formats
SELECT s.store_format,
       CASE WHEN t.transaction_date < DATE '2025-07-01'
        THEN 'H1' 
        ELSE 'H2' 
        END AS half,
       COUNT(*) AS n, ROUND(AVG(t.basket_value_zar),2) AS avg_basket
FROM freshmart_analysis.default.fresh_mart_transactiondataset_1 t
JOIN freshmart_analysis.default.fresh_mart_storesdataset_1 s ON t.store_id = s.store_id
GROUP BY s.store_format, half;

--Decline by Province
SELECT s.province,
       CASE WHEN t.transaction_date < DATE '2025-07-01' THEN 'H1' ELSE 'H2' END AS half,
       ROUND(AVG(t.basket_value_zar),2) AS avg_basket
FROM freshmart_analysis.default.fresh_mart_transactiondataset_1 t
JOIN freshmart_analysis.default.fresh_mart_storesdataset_1 s ON t.store_id = s.store_id
GROUP BY s.province, half;

--competitor analysis
SELECT
    CASE WHEN t.transaction_date < s.competitor_open_date 
    THEN 'Before' 
    ELSE 'After' 
    END AS period,
    COUNT(*) AS n, ROUND(AVG(t.basket_value_zar),2) AS avg_basket
FROM freshmart_analysis.default.fresh_mart_transactiondataset_1 t
JOIN freshmart_analysis.default.fresh_mart_storesdataset_1 s ON t.store_id = s.store_id
WHERE s.has_nearby_competitor = True
  AND t.transaction_date BETWEEN s.competitor_open_date - INTERVAL 60 DAY
                              AND s.competitor_open_date + INTERVAL 60 DAY
GROUP BY period;

--stockouts
SELECT
    CASE WHEN so.stockout_id IS NULL THEN 'Normal weekend'
     ELSE 'Stockout weekend' 
     END AS day_status,
    COUNT(*) AS n_transactions,
    ROUND(AVG(t.basket_value_zar),2) AS avg_basket
FROM freshmart_analysis.default.fresh_mart_transactiondataset_1 t
LEFT JOIN freshmart_analysis.default.fresh_mart_stockoutsdataset_1 so ON t.store_id = so.store_id AND t.transaction_date = so.date
WHERE t.store_id IN ('ST1002','ST1006','ST1039','ST1024','ST1041','ST1040')
  AND t.day_of_week IN ('Saturday','Sunday')
GROUP BY day_status;